# What did a 1% false-accept budget cost on FMA Large?
LSH retained 328 of the 691 correct decisions made by the frozen score-and-gap rule. Nine other clips became correct, bringing the total to 337. False accepts among absent tracks fell from 23 to eight.

Run all cells with Python 3, pandas and Jupyter. This CPU replay reads the committed GPU scores and thresholds. The [experiment report](../reports/large-recalibration.md) gives the GPU reproduction steps and song-level confidence intervals.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'reports/large-recalibration.json').is_file())
reports = root / 'reports'
experiment = json.loads((reports / 'large-recalibration.json').read_text(encoding='utf-8'))
scores = pd.read_csv(reports / 'large-recalibration-scores.csv.gz', converters={'top_score': float, 'gap': float})
assert not scores.duplicated(['method', 'phase', 'track_id', 'family']).any()
assert scores.groupby(['method', 'phase', 'known']).size().eq(1024).all()
assert set(scores.loc[scores.phase == 'development', 'track_id']).isdisjoint(
    scores.loc[scores.phase == 'test', 'track_id'])
scores.groupby(['phase', 'method']).agg(clips=('track_id', 'size'), songs=('track_id', 'nunique'))

clips  songs
phase       method              
development exact    2048   1024
            lsh      2048   1024
test        exact    2048   1024
            lsh      2048   1024

## Replay the paired decisions
Each phase has two distorted clips per song. A known query counts as correct only when the selected track ID equals the source ID and the rule accepts it. For an absent source ID, any acceptance counts against the budget. Compare both rules on the very same query rows to distinguish retained matches from new ones.

In [2]:
def accepted(rows, thresholds):
    mask = rows.top_score >= thresholds['score_threshold']
    gap = thresholds['gap_threshold']
    return mask if gap is None else mask & (rows.gap >= gap)

results = []
for method, report in experiment['methods'].items():
    test = scores[(scores.method == method) & (scores.phase == 'test')]
    development = scores[(scores.method == method) & (scores.phase == 'development')]
    correct_id = test.known & test.top_track.eq(test.track_id)
    for rule, comparison in report['comparison'].items():
        tuned = comparison['large_recalibrated']
        dev_false = (accepted(development, tuned) & ~development.known).sum()
        assert dev_false == tuned['development_unknown_false_accepts'] == 10
        before = accepted(test, experiment['frozen_thresholds'][rule])
        after = accepted(test, tuned)
        old, new = before & correct_id, after & correct_id
        kept, lost, gained = (old & new).sum(), (old & ~new).sum(), (~old & new).sum()
        assert [kept, lost, gained] == [comparison['known_correct_' + key]
                                        for key in ('retained', 'lost', 'gained')]
        old_false, new_false = (before & ~test.known).sum(), (after & ~test.known).sum()
        assert new_false == tuned['unknown_false_accepts']
        results.append([method.upper(), rule.replace('_', ' '), old.sum(), kept, lost, gained,
                        new.sum(), old_false, new_false])

pd.DataFrame(results, columns=['Search', 'Frozen rule', 'Correct before', 'Retained', 'Lost',
                               'Gained', 'Correct after', 'False accepts before', 'False accepts after'])

,Search,Frozen rule,Correct before,Retained,Lost,Gained,Correct after,False accepts before,False accepts after
0,EXACT,score only,523,336,187,0,336,18,8
1,EXACT,score and gap,691,328,363,8,336,23,8
2,LSH,score only,526,337,189,0,337,18,8
3,LSH,score and gap,691,328,363,9,337,23,8


The development search selected a score cutoff of about 0.886316 for each retriever and no minimum gap. On held-out absent clips, 8/1,024 gives 0.78%. The 95% song-bootstrap interval spans 0.29–1.37%, so a population false-accept rate below 1% remains uncertain.

The strict ID definition matters. Four of the ten audited development false accepts reproduced the same clean waveform under a different track ID after mono conversion to 4 kHz. See the [development error audit and research comparison](../reports/large-fingerprint-research.md).